# Evolve: ES vs Gradient Benchmarks on Colab

Two experiments:
1. **GPU Benchmark** — EGGROLL ES vs GRPO/SFT on Qwen2.5-1.5B (countdown task)
2. **MoE Routing** — ES vs gradient router optimization on OLMoE-1B-7B

**Requirements:** Colab with T4 GPU (free tier works).

Runtime > Change runtime type > T4 GPU

## Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone repo
!git clone https://github.com/TomOffermann/evolve.git 2>/dev/null || (cd evolve && git pull)
%cd evolve

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes numpy

---
## Experiment 1: MoE Routing (ES vs Gradient)

Tests whether ES can optimize hard routing decisions better than gradient methods.

**Key insight:** Gradient methods train with soft routing (softmax) but deploy with
hard routing (argmax). ES directly optimizes the hard routing decision.

In [ ]:
# Quick smoke test (~5 min)
!python code/experiments/moe_routing/run_olmoe.py \
    --model allenai/OLMoE-1B-7B-0924 \
    --load-in-8bit \
    --task arc \
    --quick

In [ ]:
# Full MoE routing experiment (~30-60 min)
!python code/experiments/moe_routing/run_olmoe.py \
    --model allenai/OLMoE-1B-7B-0924 \
    --load-in-8bit \
    --task arc \
    --n-pop 32 \
    --generations 100 \
    --output moe_routing_olmoe_full.json

---
## Experiment 2: GPU Benchmark (ES vs GRPO/SFT)

Head-to-head comparison on the countdown task using a real pretrained LM.

This validates the CPU benchmark findings at scale: partitioned ES should
maintain performance while vanilla ES degrades.

In [ ]:
# Quick smoke test with SmolLM2-135M (~10 min)
!python code/benchmark/run.py \
    --model HuggingFaceTB/SmolLM2-135M \
    --methods eggroll_vanilla,eggroll_partitioned \
    --seeds 0 \
    --quick

In [ ]:
# Full benchmark with Qwen2.5-1.5B (~2-4 hours)
# ES methods only (GRPO needs more memory)
!python code/benchmark/run.py \
    --model Qwen/Qwen2.5-1.5B \
    --dtype bfloat16 \
    --methods eggroll_vanilla,eggroll_partitioned,eggroll_selective \
    --seeds 0,1,2 \
    --generations 200

In [ ]:
# GRPO baseline (separate — needs reference model copy)
!python code/benchmark/run.py \
    --model Qwen/Qwen2.5-1.5B \
    --dtype bfloat16 \
    --methods grpo \
    --seeds 0 \
    --generations 200

In [ ]:
# View results
import json
from pathlib import Path

for f in sorted(Path('code/benchmark/results').glob('*.json')):
    r = json.load(open(f))
    print(f"{r['method']:25s} pass@1={r['final'].get('pass@1',0):.4f}  "
          f"pass@16={r['final'].get('pass@16',0):.4f}  "
          f"ppl_delta={r.get('forgetting',{}).get('perplexity_delta',0):+.1f}  "
          f"wall={r['wall_seconds']:.0f}s")

---
## Download Results

Download result JSONs to your local machine.

In [ ]:
# Download all results
from google.colab import files
import glob

for f in glob.glob('code/benchmark/results/*.json') + glob.glob('moe_routing_*.json'):
    files.download(f)